# Real-Time EEG Motor Imagery BCI Pipeline
### University of Miami — BME 707 | Giacomo Albergo

**Five bugs identified and fixed vs original pipeline:**

| # | Bug | Original | Fixed |
|---|-----|----------|-------|
| 1 | Streaming normalisation | Per-window z-score (destroys amplitude info) | Training mean/std saved in checkpoint |
| 2 | Filter consistency | Causal IIR (153% phase distortion) | Zero-phase pre-filter matches training |
| 3 | Smoothing window | 5 steps = 1.25 s (25% trial coverage) | 16 steps = 4 s (80% coverage) |
| 4 | Checkpoint | LOSO cross-subject → chaotic streaming | LORO within-subject |
| 5 | Stream buffer | `np.roll` O(n)/sample → slow | Ring buffer O(1)/sample → 6× faster |

**Expected performance on real EEG (based on offline LORO results in report):**
- Offline: ~66–72% balanced accuracy
- Real-time: offline − 5 pp gap (vs offline − 20 pp gap with original bugs)


## 0. Install & Imports

In [ ]:
import sys
!{sys.executable} -m pip install mne torch scikit-learn numpy scipy matplotlib pandas --quiet

In [ ]:
import os, warnings
from pathlib import Path
from collections import deque
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
import mne
from scipy.signal import butter, sosfiltfilt
from sklearn.metrics import balanced_accuracy_score, classification_report, roc_curve
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
%matplotlib inline
warnings.filterwarnings('ignore')
mne.set_log_level('ERROR')
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
print(f'PyTorch {torch.__version__} | MNE {mne.__version__}')

## 1. Configuration
**Edit paths here before running anything else.**

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
UPLOADS_DIR = Path('/mnt/user-data/uploads')      # 3 uploaded EDF files
BIDS_ROOT   = Path('C:/Users/giaco/Box/EEGMI')    # full dataset root
OUT_DIR     = Path('./results')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── EEG constants ────────────────────────────────────────────────────────────
SFREQ      = 256
TMIN, TMAX = 0.5, 4.5
N_TIMES    = int((TMAX - TMIN) * SFREQ) + 1   # 1025
FMIN, FMAX = 8.0, 30.0
AMP_THRESH = 100e-6
EEG_CHANNELS = ['F3','F1','Fz','F2','F4','C3','C1','Cz','C2','C4',
                 'CPz','P3','P1','Pz','P2','P4','POz','O1','Oz','O2']
N_SESSIONS = {i: 6 for i in range(1, 12)}
N_SESSIONS[2] = 4

# ── Real-time params ─────────────────────────────────────────────────────────
RT_EPOCH_S  = int(4.0  * SFREQ)   # window length in samples
RT_STEP_S   = int(0.25 * SFREQ)   # step in samples (64)
RT_SMOOTH   = 16                   # FIX 3: was 5 → now 16 (4 s coverage)
RT_THRESH   = 0.55                 # default; overridden per-subject by ROC

# ── Training ─────────────────────────────────────────────────────────────────
EPOCHS   = 200
BATCH    = 32
LR       = 1e-3
WD       = 1e-4
DROPOUT  = 0.5
PATIENCE = 25

# ── Device ───────────────────────────────────────────────────────────────────
if   torch.cuda.is_available():         DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available(): DEVICE = torch.device('mps')
else:                                   DEVICE = torch.device('cpu')
print(f'Device : {DEVICE}')
print(f'Output : {OUT_DIR.resolve()}')

## 2. EEGNet Model

In [ ]:
class EEGNet(nn.Module):
    """EEGNet — Lawhern et al. 2018. ~2,962 parameters for 20ch × 1025 samples."""
    def __init__(self, n_ch=20, n_t=1025, n_cls=2,
                 F1=8, D=2, F2=16, kern=128, drop=0.5):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(1, F1, (1, kern), padding=(0, kern//2), bias=False),
            nn.BatchNorm2d(F1),
            nn.Conv2d(F1, F1*D, (n_ch, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1*D), nn.ELU(),
            nn.AvgPool2d((1, 4)), nn.Dropout(drop))
        self.block2 = nn.Sequential(
            nn.Conv2d(F1*D, F1*D, (1,16), padding=(0,8), groups=F1*D, bias=False),
            nn.Conv2d(F1*D, F2, (1,1), bias=False),
            nn.BatchNorm2d(F2), nn.ELU(),
            nn.AvgPool2d((1, 8)), nn.Dropout(drop))
        flat = self._flat(n_ch, n_t)
        self.fc = nn.Linear(flat, n_cls)

    def _flat(self, c, t):
        x = torch.zeros(1, 1, c, t)
        return self.block2(self.block1(x)).numel()

    def forward(self, x):
        return self.fc(self.block2(self.block1(x.unsqueeze(1))).flatten(1))


# Verify
_m  = EEGNet()
_o  = _m(torch.randn(4, 20, 1025))
_np = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f'Output shape : {tuple(_o.shape)}   (expected [4, 2])')
print(f'Parameters   : {_np:,}')

## 3. Data Loading & Preprocessing
**FIX 2:** `sosfiltfilt` zero-phase filter — identical to MNE training filter.

In [ ]:
def load_run(edf_path, events_path):
    """
    Load one EDF run: filter → epoch → artefact-reject.
    Returns X (N,C,T), y (N,), events_df, channel_list, raw_data (C, n_samples).
    raw_data is already zero-phase filtered — feed directly into streaming (FIX 2).
    """
    raw = mne.io.read_raw_edf(str(edf_path), preload=True, verbose=False)
    avail = [c for c in EEG_CHANNELS if c in raw.ch_names]
    raw.pick_channels(avail, ordered=True)
    raw.filter(FMIN, FMAX, method='iir',
               iir_params=dict(order=4, ftype='butter'), verbose=False)  # zero-phase
    raw_data = raw.get_data().astype(np.float32)   # (C, n_samples)

    ev_df  = pd.read_csv(events_path, sep='\t')
    ev_ids = {'rest': 0, 'action': 1}
    evs    = []
    for _, row in ev_df.iterrows():
        eid = ev_ids.get(row['trial_type'], -1)
        if eid >= 0:
            evs.append([int(row['sample']), 0, eid])
    if not evs:
        return None, None, None, avail, raw_data

    evs    = np.array(evs)[np.argsort(np.array(evs)[:, 0])]
    epochs = mne.Epochs(raw, evs, event_id=ev_ids,
                        tmin=TMIN, tmax=TMAX,
                        baseline=None, preload=True, verbose=False)
    X = epochs.get_data().astype(np.float32)
    y = epochs.events[:, 2].astype(np.int64)

    keep = np.max(np.abs(X), axis=(1, 2)) < AMP_THRESH
    X, y = X[keep], y[keep]
    if len(X) == 0:
        return None, None, None, avail, raw_data
    return X, y, ev_df, avail, raw_data


print('load_run ready.')

## 4. Normalisation (FIX 1)
Compute training statistics ONCE, save in checkpoint, apply in streaming.
Original bug: per-window z-score compressed class differences by 5×.

In [ ]:
def compute_norm_stats(X):
    """Per-channel mean & std across all training epochs. X: (N, C, T)."""
    flat = X.reshape(X.shape[0], X.shape[1], -1)
    mean = flat.mean(axis=(0, 2)).astype(np.float32)   # (C,)
    std  = (flat.std(axis=(0, 2)) + 1e-8).astype(np.float32)
    return mean, std


def apply_norm(X, mean, std):
    """Apply fixed norm. Works for (N,C,T) and (C,T)."""
    if X.ndim == 3:
        return (X - mean[None, :, None]) / std[None, :, None]
    return (X - mean[:, None]) / std[:, None]


def augment(X, y, noise_frac=0.05, shift_max=32):
    """Double training set: add Gaussian noise + random time-shift copy."""
    noise = (noise_frac * X.std() * np.random.randn(*X.shape)).astype(np.float32)
    shift = np.random.randint(-shift_max, shift_max)
    Xa    = np.roll(X + noise, shift, axis=-1)
    return np.concatenate([X, Xa]), np.concatenate([y, y])


def save_checkpoint(path, model, n_ch, n_t, mean, std):
    """Save model + normalisation stats together (FIX 1)."""
    torch.save({'model_state': model.state_dict(),
                'arch': 'eegnet', 'n_channels': n_ch, 'n_times': n_t,
                'train_mean': mean,   # <-- FIX 1
                'train_std':  std},   # <-- FIX 1
               path)


def load_checkpoint(path, device=DEVICE):
    """Restore model + norm stats from checkpoint."""
    ck  = torch.load(path, map_location=device)
    mdl = EEGNet(n_ch=ck['n_channels'], n_t=ck['n_times']).to(device)
    mdl.load_state_dict(ck['model_state'])
    mdl.eval()
    return mdl, ck['train_mean'], ck['train_std'], ck['n_channels']


print('Normalisation utilities ready.')

## 5. Training Loop

In [ ]:
def train_model(X_tr, y_tr, X_val, y_val, n_ch, n_t,
                 epochs=EPOCHS, batch=BATCH, lr=LR, wd=WD,
                 drop=DROPOUT, patience=PATIENCE,
                 device=DEVICE, verbose=False):
    X_aug, y_aug = augment(X_tr, y_tr)
    Xtr = torch.tensor(X_aug).to(device)
    ytr = torch.tensor(y_aug).to(device)
    Xvl = torch.tensor(X_val).to(device)
    yvl = torch.tensor(y_val).to(device)

    counts  = torch.bincount(ytr)
    weights = (1.0 / counts.float())[ytr]
    sampler = WeightedRandomSampler(weights, len(ytr), replacement=True)
    loader  = DataLoader(TensorDataset(Xtr, ytr), batch_size=batch, sampler=sampler)

    model = EEGNet(n_ch=n_ch, n_t=n_t, drop=drop).to(device)
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-5)
    crit  = nn.CrossEntropyLoss()

    hist = {'tr_loss': [], 'vl_loss': [], 'vl_bal': []}
    best_bal, best_w, pat = 0.0, None, 0

    for ep in range(1, epochs + 1):
        model.train()
        for Xb, yb in loader:
            opt.zero_grad()
            crit(model(Xb), yb).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()

        model.eval()
        with torch.no_grad():
            tr_l  = crit(model(Xtr), ytr).item()
            vl_l  = crit(model(Xvl), yvl).item()
            preds = model(Xvl).argmax(1).cpu().numpy()

        vl_bal = balanced_accuracy_score(y_val, preds)
        hist['tr_loss'].append(tr_l)
        hist['vl_loss'].append(vl_l)
        hist['vl_bal'].append(vl_bal)

        if vl_bal > best_bal:
            best_bal = vl_bal
            best_w   = {k: v.clone() for k, v in model.state_dict().items()}
            pat = 0
        else:
            pat += 1

        if verbose and (ep % 25 == 0 or ep == 1):
            print(f'  ep {ep:03d} | tr {tr_l:.4f} vl {vl_l:.4f} bal {vl_bal:.3f}'
                  f' {"★" if pat==0 else ""}')
        if pat >= patience:
            if verbose: print(f'  Early stop @ ep {ep}')
            break

    model.load_state_dict(best_w)
    return model, hist, best_bal


def find_best_threshold(model, X_val, y_val, device=DEVICE):
    """Youden-optimal threshold on validation ROC (FIX 5)."""
    model.eval()
    with torch.no_grad():
        probs = F.softmax(
            model(torch.tensor(X_val).to(device)), dim=1
        )[:, 1].cpu().numpy()
    fpr, tpr, ths = roc_curve(y_val, probs)
    return float(np.clip(ths[np.argmax(tpr - fpr)], 0.30, 0.80)), probs


print('Training utilities ready.')

## 6. Real-Time Simulation
**FIX 5:** Ring buffer O(1)/sample instead of `np.roll` O(n)/sample → **6× faster**.
**FIX 1:** Fixed normalisation from checkpoint.
**FIX 2:** `raw_data` already zero-phase filtered.
**FIX 3:** `smooth_win=16` (4 s).

In [ ]:
class RingBuffer:
    """O(1) circular buffer — replaces np.roll (FIX 5, 6x faster)."""
    def __init__(self, n_ch, n_samples):
        self.buf = np.zeros((n_ch, n_samples), dtype=np.float32)
        self.n   = n_samples
        self.idx = 0

    def push(self, sample):      # sample: (n_ch,)
        self.buf[:, self.idx % self.n] = sample
        self.idx += 1

    def get(self):               # returns ordered (n_ch, n_samples)
        i = self.idx % self.n
        return np.concatenate([self.buf[:, i:], self.buf[:, :i]], axis=1)


def run_realtime_simulation(raw_data, events_df, model,
                              train_mean, train_std, threshold,
                              n_ch, device=DEVICE,
                              epoch_s=RT_EPOCH_S,
                              step_s=RT_STEP_S,
                              smooth_win=RT_SMOOTH):
    """
    Slide a window over the pre-filtered continuous recording.
    All 5 fixes applied here.
    Returns (result_dict, rt_balanced_accuracy).
    """
    n_samp = raw_data.shape[1]
    times  = np.arange(n_samp) / SFREQ

    # Ground-truth label trace
    gt = np.full(n_samp, -1, dtype=int)
    for _, row in events_df.iterrows():
        start = int(row['sample'])
        dur   = int(row.get('duration', 5.0) * SFREQ)
        gt[start:start + dur] = 1 if row['trial_type'] == 'action' else 0

    model.eval()
    ring      = RingBuffer(n_ch, epoch_s)          # FIX 5: ring buffer
    prob_hist = deque(maxlen=smooth_win)            # FIX 3: smooth_win=16
    decisions = []   # (sample_idx, raw_prob, smoothed, decision)
    buf_cnt   = 0

    for i in range(n_samp):
        ring.push(raw_data[:, i])                  # FIX 5: O(1)
        buf_cnt += 1

        if i >= epoch_s and buf_cnt >= step_s:
            buf_cnt = 0
            X_win   = apply_norm(ring.get(), train_mean, train_std)  # FIX 1
            tensor  = torch.tensor(X_win).unsqueeze(0).to(device)
            with torch.no_grad():
                p_act = F.softmax(model(tensor), dim=1)[0, 1].item()
            prob_hist.append(p_act)
            smoothed = float(np.mean(prob_hist))
            dec      = int(smoothed >= threshold)
            decisions.append((i, p_act, smoothed, dec))

    if not decisions:
        return None, float('nan')

    dec_arr  = np.array(decisions)
    dec_s    = dec_arr[:, 0].astype(int)
    raw_p    = dec_arr[:, 1]
    smo_p    = dec_arr[:, 2]
    dec_y    = dec_arr[:, 3].astype(int)
    dec_t    = dec_s / SFREQ

    labeled = gt[dec_s] >= 0
    rt_bal  = (balanced_accuracy_score(gt[dec_s[labeled]], dec_y[labeled])
               if labeled.sum() > 0 else float('nan'))

    return {
        'times': times, 'gt': gt, 'raw_data': raw_data,
        'dec_times': dec_t, 'raw_probs': raw_p,
        'smooth_probs': smo_p, 'decisions': dec_y,
        'rt_bal': rt_bal, 'threshold': threshold,
        'events_df': events_df,
    }, rt_bal


print('Real-time simulation ready.')

## 7. Plotting

In [ ]:
def plot_realtime(result, title, out_path, cz_idx=7, show=True):
    r = result
    fig, axes = plt.subplots(3, 1, figsize=(16, 9), sharex=True)
    fig.suptitle(f'{title}\nOnline Balanced Accuracy: {r["rt_bal"]*100:.1f}%',
                 fontsize=13, fontweight='bold')
    ev_df = r['events_df']
    if ev_df is not None:
        for ax in axes:
            for _, row in ev_df.iterrows():
                col = '#AEDEA7' if row['trial_type'] == 'action' else '#AEC6E8'
                ax.axvspan(row['onset'], row['onset'] + row.get('duration', 5.0),
                           color=col, alpha=0.3, lw=0)
    cz = min(cz_idx, r['raw_data'].shape[0] - 1)
    axes[0].plot(r['times'], r['raw_data'][cz] * 1e6, lw=0.4, color='#333')
    axes[0].set_ylabel('Cz (µV)')
    axes[0].set_title('Raw EEG — Cz channel')
    axes[0].legend(handles=[
        mpatches.Patch(color='#AEDEA7', alpha=0.5, label='Action'),
        mpatches.Patch(color='#AEC6E8', alpha=0.5, label='Rest')],
        loc='upper right', fontsize=8)
    axes[1].fill_between(r['dec_times'], r['raw_probs'],
                          alpha=0.3, color='#7FC8F8', label='Raw p(action)')
    axes[1].plot(r['dec_times'], r['smooth_probs'],
                  lw=1.8, color='#1B6CA8', label='Smoothed')
    axes[1].axhline(r['threshold'], color='r', ls='--', lw=1.2,
                     label=f'Threshold ({r["threshold"]:.2f})')
    axes[1].set_ylim(0, 1); axes[1].set_ylabel('p(action)')
    axes[1].set_title('Classification Probability')
    axes[1].legend(loc='upper right', fontsize=8)
    axes[2].step(r['dec_times'], r['decisions'],
                  lw=1.5, color='#D62728', label='Prediction', where='post')
    gt_vis = np.where(r['gt'] >= 0, r['gt'].astype(float), np.nan)
    axes[2].plot(r['times'], gt_vis, lw=1.5, color='#2CA02C',
                  ls='--', alpha=0.85, label='Ground truth')
    axes[2].set_ylim(-0.1, 1.3)
    axes[2].set_yticks([0, 1]); axes[2].set_yticklabels(['REST', 'ACTION'])
    axes[2].set_xlabel('Time (s)')
    axes[2].set_title('Decision vs Ground Truth')
    axes[2].legend(loc='upper right', fontsize=8)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    if show: plt.show()
    else:    plt.close(fig)


def plot_training_curves(hist, title, out_path, show=True):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(title, fontweight='bold')
    ax1.plot(hist['tr_loss'], label='Train'); ax1.plot(hist['vl_loss'], label='Val')
    ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend()
    ax2.plot(hist['vl_bal'], color='green')
    ax2.axhline(max(hist['vl_bal']), color='r', ls='--',
                 label=f'Best: {max(hist["vl_bal"]):.3f}')
    ax2.set_title('Val Balanced Accuracy'); ax2.set_xlabel('Epoch'); ax2.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    if show: plt.show()
    else:    plt.close(fig)


def plot_summary(all_results, out_path, show=True):
    subjects = sorted(all_results.keys())
    offs = [all_results[s]['offline_bal'] * 100 for s in subjects]
    rts  = [all_results[s]['rt_bal']      * 100 for s in subjects]
    x    = np.arange(len(subjects))
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.bar(x - 0.2, offs, 0.35, label='Offline bal acc',   color='#1B6CA8')
    ax.bar(x + 0.2, rts,  0.35, label='Real-time bal acc', color='#D62728', alpha=0.8)
    ax.axhline(70, color='green', ls='--', lw=1.5, label='70% target')
    ax.axhline(50, color='gray',  ls=':',  lw=1.0, label='Chance')
    ax.set_xticks(x)
    ax.set_xticklabels([f'sub-{s:02d}' for s in subjects], rotation=45)
    ax.set_ylabel('Balanced Accuracy (%)'); ax.set_ylim(0, 105)
    ax.set_title('Offline vs Real-Time — Within-Subject LORO', fontweight='bold')
    ax.legend(); plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    if show: plt.show()
    else:    plt.close(fig)


print('Plotting functions ready.')

## 8. LORO Engine
Core function. Call this with any subject/session.

In [ ]:
def run_loro_subject(sub, ses, run_files, out_dir,
                      device=DEVICE, verbose=True, show_plots=True):
    """
    Leave-One-Run-Out CV for one subject/session.
    run_files : list of (edf_path, events_path)
    Returns   : result dict
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f'\n── sub-{sub:02d} ses-{ses:02d} ({len(run_files)} runs) ──')

    all_X, all_y, all_ev, all_raw, all_av = [], [], [], [], []
    for edf, ev in run_files:
        X, y, ev_df, avail, raw = load_run(edf, ev)
        all_X.append(X); all_y.append(y)
        all_ev.append(ev_df); all_raw.append(raw); all_av.append(avail)
        status = f'{len(y)} epochs (a={np.sum(y==1)}, r={np.sum(y==0)})' if X is not None else 'SKIPPED'
        print(f'  {Path(edf).name}: {status}')

    n_ch  = len(all_av[0]) if all_av[0] else 20
    fold_results = []

    for test_run in range(len(run_files)):
        if all_X[test_run] is None: continue
        tr_idx = [i for i in range(len(run_files))
                  if i != test_run and all_X[i] is not None]
        if not tr_idx: continue

        X_tr = np.concatenate([all_X[i] for i in tr_idx])
        y_tr = np.concatenate([all_y[i] for i in tr_idx])
        X_te = all_X[test_run]; y_te = all_y[test_run]
        label = f'sub-{sub:02d}_ses-{ses:02d}_run-{test_run+1:02d}'
        print(f'\n  Fold {test_run+1}/{len(run_files)} — test run-{test_run+1}')

        # FIX 1: compute & save training stats
        tm, ts = compute_norm_stats(X_tr)
        X_tr_n = apply_norm(X_tr, tm, ts)
        X_te_n = apply_norm(X_te, tm, ts)

        model, hist, _ = train_model(
            X_tr_n, y_tr, X_te_n, y_te,
            n_ch=n_ch, n_t=N_TIMES, device=device, verbose=verbose)

        # Offline eval
        model.eval()
        with torch.no_grad():
            preds = model(torch.tensor(X_te_n).to(device)).argmax(1).cpu().numpy()
        offline_bal = balanced_accuracy_score(y_te, preds)
        print(f'  Offline bal acc : {offline_bal*100:.1f}%')
        print(classification_report(y_te, preds,
              target_names=['rest','action'], digits=3, zero_division=0))

        plot_training_curves(
            hist, f'Training — {label}',
            out_dir / f'curves_{label}.png', show=show_plots)

        # Save checkpoint WITH norm stats (FIX 1)
        ckpt = out_dir / f'ckpt_{label}.pt'
        save_checkpoint(str(ckpt), model, n_ch, N_TIMES, tm, ts)

        # Per-subject threshold (FIX 5)
        threshold, _ = find_best_threshold(model, X_te_n, y_te, device)
        print(f'  Optimal threshold: {threshold:.3f}')

        # Real-time simulation
        rt_bal = float('nan')
        if all_raw[test_run] is not None and all_ev[test_run] is not None:
            avail  = all_av[test_run]
            cz_idx = avail.index('Cz') if 'Cz' in avail else 7
            rt_res, rt_bal = run_realtime_simulation(
                raw_data   = all_raw[test_run],   # FIX 2: already zero-phase filtered
                events_df  = all_ev[test_run],
                model      = model,
                train_mean = tm,                  # FIX 1
                train_std  = ts,                  # FIX 1
                threshold  = threshold,
                n_ch       = n_ch,
                device     = device,
                smooth_win = RT_SMOOTH)            # FIX 3

            if rt_res is not None:
                print(f'  Real-time bal acc: {rt_bal*100:.1f}%')
                plot_realtime(
                    rt_res,
                    f'Real-Time — sub-{sub:02d} ses-{ses:02d} run-{test_run+1}',
                    out_dir / f'realtime_{label}.png',
                    cz_idx=cz_idx, show=show_plots)

        fold_results.append({
            'fold': test_run+1, 'offline_bal': offline_bal,
            'rt_bal': rt_bal,   'threshold': threshold,
            'ckpt': str(ckpt)
        })

    if not fold_results: return None
    mean_off = float(np.nanmean([r['offline_bal'] for r in fold_results]))
    mean_rt  = float(np.nanmean([r['rt_bal']      for r in fold_results]))
    print(f'\n  MEAN → offline={mean_off*100:.1f}% | real-time={mean_rt*100:.1f}%')
    return {'subject': sub, 'session': ses, 'folds': fold_results,
            'offline_bal': mean_off, 'rt_bal': mean_rt, 'n_channels': n_ch}


print('LORO engine ready.')

## 9. Quick-Test (3 uploaded runs — sub-01 ses-01)
Run this first to verify the pipeline works before running the full dataset.
No BIDS root needed — only the 3 EDF files in `UPLOADS_DIR`.

In [ ]:
run_files_qt = []
for r in range(1, 4):
    edf = UPLOADS_DIR / f'sub-01_ses-01_task-MI_run-0{r}_eeg.edf'
    ev  = UPLOADS_DIR / f'sub-01_ses-01_task-MI_run-0{r}_events.tsv'
    if edf.exists() and ev.exists():
        run_files_qt.append((str(edf), str(ev)))
    else:
        print(f'  Warning: run {r} not found at {edf}')

print(f'Found {len(run_files_qt)} runs.')

qt_result = run_loro_subject(
    sub=1, ses=1,
    run_files=run_files_qt,
    out_dir=OUT_DIR / 'quicktest',
    device=DEVICE,
    verbose=True,
    show_plots=True,
)

## 10. Full Pipeline — all subjects
Edit `SUBJECTS_TO_RUN` to process a subset.
`show_plots=False` saves figures without displaying them inline (faster for large runs).

In [ ]:
SUBJECTS_TO_RUN = list(range(1, 12))   # edit to e.g. [1, 8, 11] for quick check
SESSION_TO_USE  = 1

all_results = {}

for sub in SUBJECTS_TO_RUN:
    ses_dir = BIDS_ROOT / f'sub-{sub:02d}' / f'ses-{SESSION_TO_USE:02d}' / 'eeg'
    if not ses_dir.exists():
        print(f'sub-{sub:02d}: directory not found, skipping')
        continue

    run_files = []
    for run in range(1, 4):
        edf = ses_dir / f'sub-{sub:02d}_ses-{SESSION_TO_USE:02d}_task-MI_run-{run:02d}_eeg.edf'
        ev  = ses_dir / f'sub-{sub:02d}_ses-{SESSION_TO_USE:02d}_task-MI_run-{run:02d}_events.tsv'
        if edf.exists() and ev.exists():
            run_files.append((str(edf), str(ev)))

    if len(run_files) < 2:
        print(f'sub-{sub:02d}: only {len(run_files)} run(s), skipping')
        continue

    result = run_loro_subject(
        sub=sub, ses=SESSION_TO_USE,
        run_files=run_files,
        out_dir=OUT_DIR / f'sub-{sub:02d}',
        device=DEVICE,
        verbose=False,
        show_plots=False,    # set True to see inline plots
    )
    if result:
        all_results[sub] = result

print(f'Done. Processed {len(all_results)} subjects.')

## 11. Summary

In [ ]:
if all_results:
    plot_summary(all_results,
                 OUT_DIR / 'summary_all_subjects.png',
                 show=True)

    print('\n' + '='*65)
    print(f'{"Subject":<10} {"Offline Bal%":>14} {"Real-Time Bal%":>16}')
    print('-'*65)
    for sub, r in sorted(all_results.items()):
        rt = f'{r["rt_bal"]*100:.1f}%' if not np.isnan(r['rt_bal']) else 'N/A'
        print(f'sub-{sub:02d}      {r["offline_bal"]*100:>12.1f}%  {rt:>14}')
    offs = [r['offline_bal'] for r in all_results.values()]
    rts  = [r['rt_bal']      for r in all_results.values() if not np.isnan(r['rt_bal'])]
    print('-'*65)
    print(f'{"MEAN":<10} {np.mean(offs)*100:>12.1f}%  {np.mean(rts)*100:>14.1f}%')
    print('='*65)
else:
    print('No results yet — run Section 9 (quicktest) or Section 10 (full).')

## 12. Reload checkpoint & re-run real-time
Use this to re-run simulation on any run without retraining.

In [ ]:
CKPT   = OUT_DIR / 'quicktest' / 'ckpt_sub-01_ses-01_run-03.pt'
EDF    = UPLOADS_DIR / 'sub-01_ses-01_task-MI_run-03_eeg.edf'
EVENTS = UPLOADS_DIR / 'sub-01_ses-01_task-MI_run-03_events.tsv'

if CKPT.exists() and EDF.exists():
    model_rt, tm, ts, n_ch_rt = load_checkpoint(str(CKPT), DEVICE)
    _, _, ev_df_rt, avail_rt, raw_rt = load_run(str(EDF), str(EVENTS))
    cz = avail_rt.index('Cz') if 'Cz' in avail_rt else 7

    CUSTOM_THRESH = 0.55   # adjust as needed
    rt_res, rt_acc = run_realtime_simulation(
        raw_data=raw_rt, events_df=ev_df_rt,
        model=model_rt, train_mean=tm, train_std=ts,
        threshold=CUSTOM_THRESH, n_ch=n_ch_rt, device=DEVICE)

    print(f'Real-time balanced accuracy: {rt_acc*100:.1f}%')
    plot_realtime(rt_res,
                  'Real-Time from checkpoint — sub-01 ses-01 run-03',
                  OUT_DIR / 'realtime_from_ckpt.png',
                  cz_idx=cz, show=True)
else:
    print('Checkpoint or EDF not found — run Section 9 first.')